In [4]:
import numpy as np
import pandas as pd
import random
import tqdm
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Pauli, SparsePauliOp, StabilizerState, DensityMatrix
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error, phase_damping_error, PauliError
import warnings
warnings.filterwarnings('ignore')

class CliffordSimulator:
    def __init__(self, num_qubits, num_layers):
        self.num_qubits = num_qubits
        self.num_layers = num_layers
        self.observables = self._create_observables()
        
    def _create_observables(self):
        """Create all ZZ and X observables"""
        observables = []
        for i in range(self.num_qubits):
            for j in range(i+1, self.num_qubits):
                pauli_str = ['I'] * self.num_qubits
                pauli_str[i] = 'Z'
                pauli_str[j] = 'Z'
                observables.append(Pauli(''.join(pauli_str)))
        
        # X observables (single qubit)
        for i in range(self.num_qubits):
            pauli_str = ['I'] * self.num_qubits
            pauli_str[i] = 'X'
            observables.append(Pauli(''.join(pauli_str)))
            
        return observables
    
    def qr_haar(self, N):
        """Generate Haar-random matrix for non-Clifford case"""
        A, B = np.random.normal(size=(N, N)), np.random.normal(size=(N, N))
        Z = A + 1j * B
        Q, R = np.linalg.qr(Z)
        Lambda = np.diag([R[i, i] / np.abs(R[i, i]) for i in range(N)])
        return np.dot(Q, Lambda)
    
    def generate_clifford_circuit(self, clifford=True):
        """Generate random Clifford circuit with TwoLocal structure"""
        qc = QuantumCircuit(self.num_qubits)
        
        if not clifford:
            for qubit in range(self.num_qubits):
                haar_unitary = self.qr_haar(2)
                qc.unitary(haar_unitary, [qubit])
        
        # Generate random Clifford layers
        for layer in range(self.num_layers):
            # Single-qubit random gates (Clifford)
            for qubit in range(self.num_qubits):
                # Use basic gates instead of Clifford objects to avoid compatibility issues
                gate_choice = random.choice(['h', 's', 'x', 'y', 'z', 'id'])
                if gate_choice == 'h':
                    qc.h(qubit)
                elif gate_choice == 's':
                    qc.s(qubit)
                elif gate_choice == 'x':
                    qc.x(qubit)
                elif gate_choice == 'y':
                    qc.y(qubit)
                elif gate_choice == 'z':
                    qc.z(qubit)
                # 'id' does nothing
            
            # CNOT ladder (entangling layer)
            for qubit in range(self.num_qubits):
                qc.cx(qubit, (qubit + 1) % self.num_qubits)
                
        return qc
    
    def compute_ideal_expectations(self, circuit, clifford=True):
        """Compute ideal expectation values using optimal method"""
        if clifford:
            state = StabilizerState(circuit)
            return np.array([np.real(state.expectation_value(obs)) for obs in self.observables])
        else:
            statevector = Statevector(circuit)
            return np.array([np.real(statevector.expectation_value(obs)) for obs in self.observables])
    
    def compute_noisy_expectations_exact(self, circuit, two_qubit_error_rate, model, clifford=True):
        # Create noise model with two-qubit depolarizing error on CNOT gates
        noise_model = NoiseModel()
        if model == 'depolarization':
            cnot_error = depolarizing_error(two_qubit_error_rate, 2)
            noise_model.add_all_qubit_quantum_error(cnot_error, ['cx'])
            
        elif model == 'pauli':
            two_qubit_error_rate = error_rate/2
            p_id = 1 - two_qubit_error_rate
            p_x = two_qubit_error_rate/3
            p_y = p_x - p_x/3
            p_z = p_x + p_x/3
            pauli_strings = ["II", "IX", "XI", "YI", "IY", "ZI", "IZ", "XX", "YY", "ZZ", "XY", "YX", "XZ", "ZX", "YZ", "ZY"]
            probas = [p_id**2, p_id*p_x, p_id*p_x, p_id*p_y, p_id*p_y, p_id*p_z, p_id*p_z, p_x**2, p_y**2, p_z**2, p_x*p_y, p_x*p_y, p_x*p_z, p_x*p_z, p_y*p_z, p_y*p_z]
            pauli_error = PauliError(pauli_strings, probas)
            noise_model.add_all_qubit_quantum_error(pauli_error, ['cx'])
            
        elif model == 'composite':
            amp_err = amplitude_damping_error(two_qubit_error_rate / 2)
            phase_err = phase_damping_error(two_qubit_error_rate / 2)
            two_qubit_amp = amp_err.tensor(amp_err)
            two_qubit_phase = phase_err.tensor(phase_err)
            dep_err = depolarizing_error(two_qubit_error_rate, 2)
            combined_error = two_qubit_amp.compose(two_qubit_phase).compose(dep_err)
            noise_model.add_all_qubit_quantum_error(combined_error, 'cx')
        
        # Use density matrix simulator for noisy simulation
        simulator = AerSimulator(method='density_matrix', noise_model=noise_model)
        
        # Add save_density_matrix instruction to the circuit
        circuit_with_save = circuit.copy()
        circuit_with_save.save_density_matrix()
        
        # Run the simulation
        result = simulator.run(circuit_with_save).result()
        
        # Extract the density matrix
        density_matrix_data = result.data(0).get('density_matrix')
        
        dm = DensityMatrix(density_matrix_data)
        expectations = [np.real(dm.expectation_value(obs)) for obs in self.observables]

        
        return np.array(expectations)

def generate_dataset(num_qubits, num_layers, num_samples, two_qubit_error_rate, model, clifford=True):
    simulator = CliffordSimulator(num_qubits, num_layers)
    df = pd.DataFrame({'noisy': [], 'target': []})
    
    for _ in tqdm.tqdm(range(num_samples)):
        # Generate random circuit
        circuit = simulator.generate_clifford_circuit(clifford)
        
        ideal_expectations = simulator.compute_ideal_expectations(circuit, clifford)
        
        noisy_expectations = simulator.compute_noisy_expectations_exact(circuit, two_qubit_error_rate, model, clifford)
        
        new_row = pd.DataFrame({
            'noisy': [noisy_expectations.tolist()], 
            'target': [ideal_expectations.tolist()]
        })
        df = pd.concat([df, new_row], ignore_index=True)
        
    return df

def save_df(df, name):
    """Save dataframe with expanded noisy and target columns"""
    for n in range(len(df.iloc[0]['noisy'])):
        df[f'noisy_{n}'] = df['noisy'].map(lambda x: x[n])

    for n in range(len(df.iloc[0]['target'])):
        df[f'target_{n}'] = df['target'].map(lambda x: x[n])

    df.to_csv(f'./data/datasets/{name}.csv', index=False)

In [ ]:
num_qubits = 12
num_layers = 4
num_samples = 1500
model = 'depolarization'

for two_qubit_error_rate in [0.01, 0.05, 0.1]:
    df = generate_dataset(
        num_qubits=num_qubits,
        num_layers=num_layers,
        num_samples=num_samples,
        two_qubit_error_rate=two_qubit_error_rate,
        model = model,
        clifford=False  # Set to False to include Haar unitaries
    )
    
    save_df(df, 'near_' + model + str(two_qubit_error_rate).replace('.', ''))
    
    df = generate_dataset(
        num_qubits=num_qubits,
        num_layers=num_layers,
        num_samples=num_samples,
        two_qubit_error_rate=two_qubit_error_rate,
        model = model,
        clifford=True  # Set to False to include Haar unitaries
    )
    
    save_df(df, model + str(two_qubit_error_rate).replace('.', ''))

 20%|████████████████▏                                                              | 307/1500 [04:58<19:14,  1.03it/s]